In [1]:
import pandas as pd
import importlib

import plotly.express as px

import irina.utility_functions as uf

In [49]:
# how interest in topics change with time on the global level

In [2]:
importlib.reload(uf)

<module 'Irina.utility_functions' from '/Users/irinavorobeva/PycharmProjects/geoTopics/Irina/utility_functions.py'>

In [7]:
df_global = pd.read_csv(uf.PATH+"df_global_comparison_yearly.csv").drop_duplicates()

In [8]:
start_year = 1970
end_year = 2023

In [15]:
df_global_topics = (
    df_global
    .merge(uf.df_topics[["subfield_id", "subfield_name", "field_name", "domain_name"]].drop_duplicates(),
           on="subfield_id", how="left")
    .assign(
        probability_individual_change=lambda df: df.groupby("subfield_id").probability_individual.transform(lambda x: (x - x.iloc[0])),
        probability_individual_relative=lambda df: df.groupby("subfield_id").probability_individual.transform(lambda x: (x / x.iloc[0])),
        probability_collab_change=lambda df: df.groupby("subfield_id").probability_collab.transform(lambda x: (x - x.iloc[0])),
        probability_collab_relative=lambda df: df.groupby("subfield_id").probability_collab.transform(lambda x: (x / x.iloc[0])),
    )
    # .query("subfield_id in @subfields_list")
)

In [22]:
df_prob_all = (
    df_global
    .pivot(index="year", columns="subfield_id", values="probability_all")
)

df_prob_indiv = (
    df_global
    .pivot(index="year", columns="subfield_id", values="probability_individual")
)

df_prob_collab = (
    df_global
    .pivot(index="year", columns="subfield_id", values="probability_collab")
)

In [24]:
uf.plotly_heatmap(
    df_prob_all,
    x_labels=df_prob_all.columns.astype(str),
    y_labels=df_prob_all.index,
    x_type="subfield",
    z_min=0, z_max=0.05,
    line_height=10,
    colorscale="non-symmetric",
    title="Probability distribution among science subfields (all)",
)

In [25]:
uf.plotly_heatmap(
    df_prob_indiv,
    x_labels=df_prob_indiv.columns.astype(str),
    y_labels=df_prob_indiv.index,
    x_type="subfield",
    z_min=0, z_max=0.05,
    line_height=10,
    colorscale="non-symmetric",
    title="Probability distribution among science subfields (individual)",
)

In [26]:
uf.plotly_heatmap(
    df_prob_collab,
    x_labels=df_prob_collab.columns.astype(str),
    y_labels=df_prob_collab.index,
    x_type="subfield",
    z_min=0, z_max=0.05,
    line_height=10,
    colorscale="non-symmetric",
    title="Probability distribution among science subfields (collaboration)",
)

In [10]:
# year = 2023
# domain_name = "Social Sciences"
# px.bar(
#     (
#         df_global_topics
#         .assign(subfield=lambda df: df.subfield_id.astype(str))
#         .query("year == @year")
#         .query(f"domain_name == @domain_name")
#     ),
#     x="counts_loss_pct",
#     y="subfield_name",
#     color="field_name",
#     height=1000,
# )

In [27]:
df_global_topics.domain_name.drop_duplicates()

0          Life Sciences
11       Social Sciences
47     Physical Sciences
140      Health Sciences
Name: domain_name, dtype: object

In [28]:
# fig = px.area(
#     (
#         df_global_topics
#         .groupby(["domain_name", "year"], as_index=False)
#         .probability_all
#         .sum()
#     ),
#     x="year",
#     y="probability_all",
#     color="domain_name",
#     labels={"probability_all": "Probability All", "year": "Year"},
#     title="Articles distribution among science domains (including collaborations)"
# )
# fig.show()
# # fig.write_image("../images/articles_by_domains_dynamics.png",
# #                 width=1200, height=600, scale=2)

In [30]:
fig = px.area(
    (
        df_global_topics
        .groupby(["domain_name", "year"], as_index=False)
        .probability_individual
        .sum()
    ),
    x="year",
    y="probability_individual",
    color="domain_name",
    labels={"probability_individual": "Probability Individual", "year": "Year"},
    title="Articles distribution among science domains (individual)"
)
fig.show()
# fig.write_image("../images/articles_by_domains_dynamics.png",
#                 width=1200, height=600, scale=2)

In [32]:
fig = px.area(
    (
        df_global_topics
        .groupby(["domain_name", "year"], as_index=False)
        .probability_collab
        .sum()
    ),
    x="year",
    y="probability_collab",
    color="domain_name",
    labels={"probability_collab": "Probability Collaborations", "year": "Year"},
    title="Articles distribution among science domains (collaborations)"
)
fig.show()
# fig.write_image("../images/articles_by_domains_dynamics.png",
#                 width=1200, height=600, scale=2)

In [48]:
year = 1980
df_long = (
    df_global_topics
    .query("year == @year")
    .melt(
        id_vars=["domain_name", "subfield_name", "subfield_id", "counts_collab_pct"],
        value_vars=[
            "probability_collab_change",
            "probability_individual_change",
        ],
        var_name="type",
        value_name="probability_change",
    )
)

df_long["type"] = df_long["type"].map({
    "probability_collab_change": "Collaborative",
    "probability_individual_change": "Individual",
})

fig = px.box(
    df_long,
    x="domain_name",
    y="probability_change",
    color="type",          # compare collab vs indiv inside each domain
    points="all",
    height=600,
    title=f"Probability change {year} vs {start_year}",
    hover_data=["subfield_name", "subfield_id"],
)
fig.show()

In [18]:
n = 10
subfield_top = (
    df_global_topics
    .query("year == @year")
    .sort_values("probability_individual_change", ascending=False)
    .head(n)
    .subfield_id
    .to_list()
)
subfield_bottom = (
    df_global_topics
    .query("year == @year")
    .sort_values("probability_individual_change", ascending=True)
    .head(n)
    .subfield_id
    .to_list()
)
subfield_stable = (
    df_global_topics
    .query("year == @year")
    .assign(abs_change = lambda df: df.probability_individual_change.abs())
    .sort_values("abs_change", ascending=True)
    .head(n)
    .subfield_id
    .to_list()
)
subfields_list = [1306, 1313, 1111, 1312, 1110, 1105,
                  3304, 3312, 2002, 1202, 3314, 1211,
                  1702, 1710, 2208, 1607, 1605, 3107,
                  3600, 2730, 2739, 2712, 2737, 3404]

In [21]:
fig = px.line(
    df_global_topics.query("subfield_id == @subfield_top"),
    x="year",
    y="probability_individual",
    color="domain_name",      # now grouped by domain in text
    line_dash="subfield_name",
    hover_name="subfield_name",
    title=f"Probability growth: top {n}",
)
fig.update_layout(
    legend=dict(
        title="Subfield:",
        orientation="h",   # horizontal
        yanchor="bottom",
        y=-0.5,           # distance below plot
        xanchor="center",
        x=0.5
    )
)

fig.show()

In [22]:
domain_name = "Health Sciences"
px.line(
    df_global_topics.query("subfield_id in @subfields_list and domain_name == @domain_name"),
    x="year",
    y="probability_individual",
    color="subfield_name",
    hover_name="subfield_name",
    title=domain_name,
)

In [26]:
(
    df_global_topics
    .query("year == @year")
    .sort_values("probability_individual", ascending=False)
    [["year", "subfield_name", "field_name", "probability_individual"]]
    .head(10)
)

,year,subfield_name,field_name,probability_individual
13552,2023,Sociology and Political Science,Social Sciences,0.035202
13425,2023,Electrical and Electronic Engineering,Engineering,0.034910
13362,2023,Molecular Biology,"Biochemistry, Genetics and Molecular Biology",0.032304
13544,2023,Education,Social Sciences,0.027781
13507,2023,Surgery,Medicine,0.023649
13387,2023,Artificial Intelligence,Computer Science,0.022269
13421,2023,Biomedical Engineering,Engineering,0.020484
13454,2023,Materials Chemistry,Materials Science,0.018805
13395,2023,Information Systems,Computer Science,0.018057
13427,2023,Mechanical Engineering,Engineering,0.016677


In [23]:
# year = 2023
# top_10_latex = (
#     df_global_topics
#     .query("year == @year")
#     .sort_values("probability_individual", ascending=False)
#     [["subfield_name", "field_name", "probability_individual"]]
#     .head(10)
# ).applymap(uf.escape_latex).to_latex(index=False)
#
# top_10_latex